# Polymarket Arbitraj Botu — Google Colab Test

Bu notebook **paper mode** çalışır — gerçek emir göndermez, sadece fırsatları gösterir.

Gerçek emir için `.env` bilgilerini doldurmak gerekir.

In [ ]:
# 1. Bağımlılıkları kur
!pip install aiohttp nest_asyncio py-clob-client python-dotenv -q

In [ ]:
# 2. Colab'da asyncio yaması (Jupyter için gerekli)
import nest_asyncio
nest_asyncio.apply()

import asyncio
import aiohttp
import time
import logging
import sys
from dataclasses import dataclass
from typing import Any

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    stream=sys.stdout,
)
log = logging.getLogger(__name__)
print("Hazır.")

In [ ]:
# 3. Ayarlar
PAPER_MODE     = True      # False yapınca gerçek emir gönderir
SCAN_INTERVAL  = 10        # saniye
MIN_VOLUME_24H = 10_000    # USDC
MIN_PROFIT     = 0.015     # %1.5 minimum kâr
MAX_TRADE_USDC = 20.0      # per leg
PAGE_LIMIT     = 500

GAMMA_URL = "https://gamma-api.polymarket.com/markets"
CLOB_HOST = "https://clob.polymarket.com"

print(f"PAPER_MODE={PAPER_MODE} | MIN_PROFIT={MIN_PROFIT*100:.1f}% | MAX_TRADE={MAX_TRADE_USDC} USDC")

In [ ]:
# 4. Veri yapıları ve yardımcı fonksiyonlar

@dataclass
class Market:
    id: str
    question: str
    yes_token_id: str
    no_token_id: str
    yes_bid: float | None
    yes_ask: float | None
    no_bid: float | None
    no_ask: float | None
    volume24h: float

@dataclass
class ArbOpportunity:
    market: Market
    direction: str
    profit_pct: float
    yes_price: float
    no_price: float

def _f(v: Any) -> float | None:
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

def parse_market(raw: dict) -> Market | None:
    vol = _f(raw.get("volume24hr")) or 0.0
    if vol < MIN_VOLUME_24H:
        return None
    tokens = raw.get("tokens") or []
    yes_tok = next((t for t in tokens if str(t.get("outcome","")).upper() == "YES"), None)
    no_tok  = next((t for t in tokens if str(t.get("outcome","")).upper() == "NO"),  None)
    if not yes_tok or not no_tok:
        return None
    yes_tid = yes_tok.get("token_id", "")
    no_tid  = no_tok.get("token_id", "")
    if not yes_tid or not no_tid:
        return None
    yes_bid = _f(raw.get("bestBid"))
    yes_ask = _f(raw.get("bestAsk"))
    no_bid  = (1.0 - yes_ask) if yes_ask is not None else None
    no_ask  = (1.0 - yes_bid) if yes_bid is not None else None
    return Market(
        id=str(raw.get("id","")),
        question=(raw.get("question") or raw.get("slug") or "?").strip(),
        yes_token_id=yes_tid,
        no_token_id=no_tid,
        yes_bid=yes_bid, yes_ask=yes_ask,
        no_bid=no_bid,   no_ask=no_ask,
        volume24h=vol,
    )

print("Fonksiyonlar yüklendi.")

In [ ]:
# 5. Market tarayıcı + CLOB doğrulaması

async def fetch_markets(session: aiohttp.ClientSession) -> list[dict]:
    markets, offset = [], 0
    while True:
        async with session.get(
            GAMMA_URL,
            params={"active":"true","closed":"false","limit":PAGE_LIMIT,"offset":offset},
        ) as r:
            r.raise_for_status()
            batch = await r.json()
        if not batch:
            break
        markets.extend(batch)
        if len(batch) < PAGE_LIMIT:
            break
        offset += PAGE_LIMIT
    return markets

async def fetch_best_prices(session, token_id: str) -> tuple[float|None, float|None]:
    try:
        async with session.get(f"{CLOB_HOST}/book", params={"token_id": token_id}) as r:
            if r.status != 200:
                return None, None
            data = await r.json()
        bids = data.get("bids") or []
        asks = data.get("asks") or []
        best_bid = max((_f(b.get("price")) for b in bids if b.get("price")), default=None)
        best_ask = min((_f(a.get("price")) for a in asks if a.get("price")), default=None)
        return best_bid, best_ask
    except Exception:
        return None, None

async def verify_opportunity(session, market: Market) -> ArbOpportunity | None:
    (yes_bid, yes_ask), (no_bid, no_ask) = await asyncio.gather(
        fetch_best_prices(session, market.yes_token_id),
        fetch_best_prices(session, market.no_token_id),
    )
    if yes_ask is not None and no_ask is not None:
        total = yes_ask + no_ask
        if total < (1.0 - MIN_PROFIT):
            return ArbOpportunity(market, "BUY", (1.0 - total)*100, yes_ask, no_ask)
    if yes_bid is not None and no_bid is not None:
        total = yes_bid + no_bid
        if total > (1.0 + MIN_PROFIT):
            return ArbOpportunity(market, "SELL", (total - 1.0)*100, yes_bid, no_bid)
    return None

def quick_screen(m: Market) -> bool:
    if m.yes_ask is not None and m.no_ask is not None:
        if (m.yes_ask + m.no_ask) < (1.0 - MIN_PROFIT/2):
            return True
    if m.yes_bid is not None and m.no_bid is not None:
        if (m.yes_bid + m.no_bid) > (1.0 + MIN_PROFIT/2):
            return True
    return False

print("Tarayıcı hazır.")

In [ ]:
# 6. Ana döngü — çalıştır (durdurmak için: Çalıştır > Çalıştırmayı Durdur)

SCAN_COUNT = 0

async def run():
    global SCAN_COUNT
    connector = aiohttp.TCPConnector(limit=50, ttl_dns_cache=300)
    headers = {"User-Agent": "polymarket-arb/colab", "Accept": "application/json"}

    async with aiohttp.ClientSession(connector=connector, headers=headers) as session:
        print(f"Bot başladı | PAPER_MODE={PAPER_MODE}")
        print("─" * 60)

        while True:
            SCAN_COUNT += 1
            t0 = time.monotonic()

            try:
                raw = await fetch_markets(session)
                markets = [m for r in raw if (m := parse_market(r))]
                candidates = [m for m in markets if quick_screen(m)]

                elapsed_ms = (time.monotonic() - t0) * 1000
                print(f"\n[Tarama #{SCAN_COUNT}] {len(markets)} market | {len(candidates)} aday | {elapsed_ms:.0f}ms")

                if candidates:
                    tasks = [verify_opportunity(session, m) for m in candidates]
                    results = await asyncio.gather(*tasks, return_exceptions=True)
                    opps = [r for r in results if isinstance(r, ArbOpportunity)]
                    opps.sort(key=lambda o: o.profit_pct, reverse=True)

                    if opps:
                        print(f"  *** {len(opps)} ARB FIRSATI BULUNDU! ***")
                        for opp in opps:
                            print(f"  [{opp.direction}] {opp.market.question[:55]}")
                            print(f"       Kâr: %{opp.profit_pct:.2f} | YES={opp.yes_price:.4f} NO={opp.no_price:.4f}")
                            print(f"       Vol24h: ${opp.market.volume24h:,.0f}")
                            if PAPER_MODE:
                                print(f"       [PAPER] Gerçek emir gönderilmedi.")
                            # Gerçek emir için: await execute_arb(client, opp, loop)
                    else:
                        print("  Doğrulanmış ARB yok.")
                else:
                    print("  ARB adayı yok.")

            except Exception as e:
                print(f"  Hata: {e}")

            await asyncio.sleep(SCAN_INTERVAL)

await run()